In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "database").is_dir() and (p / "code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook em Codigos ou em uma de suas subpastas.")
DATABASE_DIR = PROJECT_ROOT / "database"
RESULT_DIR = PROJECT_ROOT / "result" / "testes_faketweet"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Cada experimento mantém seus próprios dataframes derivados.
PROCESSED_DIR = RESULT_DIR / "processados"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "code" / "visualizacao"))


# Carrega do arquivo para o dataframe

In [16]:
import pandas as pd
strFile = str(DATABASE_DIR / "FakeTweet" / "FakeTweetBr.csv")
df_train = pd.read_csv(strFile)
df_train['classificacao'].value_counts()

classificacao
fake    188
true     91
Name: count, dtype: int64

In [17]:
!pwd

/home/joao-lucas-nogueira/Área de trabalho/Projetos/IC/Codigos/dados/FakeTweet.Br


In [18]:
df_train

,Unnamed: 0,id,subject,text,classificacao,date,retweets,favorites,permalink
0,0,1.124513e+18,macaco marielle,Marielle >BANDIDOS Narco-traficantes-Milícias ...,fake,2019-05-04 0:16,0,0,https://twitter.com/MRTT_/status/1124513050218...
1,1,1.124049e+18,macaco marielle,"Bem, as últimas noticias a respeito disso que ...",fake,2019-05-02 17:33,0,0,https://twitter.com/EvaristoKlebber/status/112...
2,5,1.119295e+18,macaco marielle,@jornalnacional convivi com notícias da Mariel...,fake,2019-04-19 14:41,0,0,https://twitter.com/luiztemper/status/11192950...
3,10,1.114583e+18,macaco marielle,"O Cesari Battisti confessou seus crimes, a esq...",fake,2019-04-06 14:36,0,0,https://twitter.com/GrimoaldoL/status/11145825...
4,11,1.113246e+18,macaco marielle,[Agência Lupa] Verificamos: É falso que Thiago...,true,2019-04-02 22:04,1,0,https://twitter.com/antmarobel/status/11132459...
...,...,...,...,...,...,...,...,...,...
274,1345,1.118650e+18,jair time personalidade,"Por “ruptura com a corrupção”, apenas um brasi...",fake,2019-04-17 20:00,0,0,https://twitter.com/kohavitt/status/1118650345...
275,1405,1.090221e+18,jair time personalidade,"Pessoal, a revista americana TIME incluiu Jair...",fake,2019-01-29 9:10,2,1,https://twitter.com/alcamorim/status/109022062...
276,1480,1.073590e+18,jair time personalidade,QUE COMECE O MIMIMI DOS BOLSOBURROS. Jair Bols...,true,2018-12-14 11:47,0,1,https://twitter.com/jorgetemer/status/10735903...
277,1499,1.072787e+18,jair time personalidade,Circula nas redes sociais uma foto da capa da ...,true,2018-12-12 6:37,0,1,https://twitter.com/juidsonjoc/status/10727874...


In [19]:
df_train['classificacao'].value_counts()

classificacao
fake    188
true     91
Name: count, dtype: int64

In [20]:
print(df_train['text'].value_counts())

text
Um dos sites anti-hoax desmentiu o boato de que o MST teria roubado vítimas do crime ambiental em Brumadinho . É muito mais fácil o MST ter AJUDADO as vítimas com doações de produtos agrícolas do que ter roubado. Quem roubou foi a Vale privatizada.                                                                                                                     2
Marielle >BANDIDOS Narco-traficantes-Milícias pisou na bola PCC >Mandante do crime #Brazão do PT #OrganizaçãoCriminosa Assassinada por THIAGO- MACACO Midia para de acusar quem destruiu a sonhada #GRANDEPATRIA @ECantanhede @CarvalhosaMo @CarlosBolsonaro @RenovMidia @roxmo @SITEFCS pic.twitter.com/8HTb0VVmj8                                                          1
Bem, as últimas noticias a respeito disso que o verdadeiro assassino de Marielle não é aqueles acusados senão um elemento de codinome  Macaco ". Ou seja, não colou a relação Bolsonaro x Marielle . Reveja suas fontes."                                            

In [21]:
from sklearn.model_selection import train_test_split

#sentences = "Sentence: " + df_train['texto-pre'].values + " Reply: " + df_train['reply'].values
sentences =  df_train['text'].values
y = df_train['classificacao'].values

# ------

In [22]:
sentences_train, sentences_test, y_train, y_test = train_test_split(sentences, y, test_size=0.1, random_state=1000)

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2))
vectorizer.fit(sentences_train)

X_train = vectorizer.transform(sentences_train)
X_test  = vectorizer.transform(sentences_test)

In [24]:
def transforma_vetor_str_int(vetor):
    vetor_retorno = []
    
    for i in range(0, len(vetor)): 
        
        if vetor[i] == 'fake':
            vetor_retorno.append(1)
        else: 
            vetor_retorno.append(2)
    return vetor_retorno

def transforma_vetor_int_str(vetor):
    vetor_retorno = []
  
    for i in range(0, len(vetor)): 
        if vetor[i] == 1:
            vetor_retorno.append('fake')
            
        if vetor[i] == 2:
            vetor_retorno.append('true')
 
    return vetor_retorno    

In [25]:
y_train2 = transforma_vetor_str_int(y_train)
y_test2 = transforma_vetor_str_int(y_test)

In [26]:
X_train.shape,len(y_train2)

((251, 7407), 251)

In [27]:
from sklearn.naive_bayes import MultinomialNB

In [28]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble  import RandomForestClassifier
from sklearn.svm import LinearSVC


cvC = 10
classifier = LogisticRegression()
pred = cross_val_predict(classifier, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier2 = SGDClassifier()
pred2 = cross_val_predict(classifier2, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier3 = BernoulliNB()
pred3 = cross_val_predict(classifier3, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier4 = RandomForestClassifier()
pred4 = cross_val_predict(classifier4, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier5 = LinearSVC()
pred5 = cross_val_predict(classifier5, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.1s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.9s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   7 out of  10 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished


In [29]:
from sklearn.metrics import accuracy_score, f1_score,precision_score,recall_score

score = accuracy_score(y_train2, pred)
score2 = accuracy_score(y_train2, pred2)
score3 = accuracy_score(y_train2, pred3)
score4 = accuracy_score(y_train2, pred4)
score5 = accuracy_score(y_train2, pred5)

In [30]:
ps1 = precision_score(y_train2, pred, average = 'macro')
r1  = recall_score(y_train2, pred, average = 'macro') 

ps2 = precision_score(y_train2, pred2, average = 'macro')
r2 = recall_score(y_train2, pred2, average = 'macro') 

ps3 = precision_score(y_train2, pred3, average = 'macro')
r3  = recall_score(y_train2, pred3, average = 'macro') 

ps4 = precision_score(y_train2, pred4, average = 'macro')
r4  = recall_score(y_train2, pred4, average = 'macro') 

ps5 = precision_score(y_train2, pred5, average = 'macro')
r5  = recall_score(y_train2, pred5, average = 'macro') 

In [31]:
f1_geral = f1_score(y_train2, pred, average = 'macro')
f1_geral2 = f1_score(y_train2, pred2, average = 'macro')
f1_geral3 = f1_score(y_train2, pred3, average = 'macro')
f1_geral4 = f1_score(y_train2, pred4, average = 'macro')
f1_geral5 = f1_score(y_train2, pred5, average = 'macro')
df1_geral = f1_score(y_train2, pred, average = None)
df1_gera2 = f1_score(y_train2, pred2, average = None)
df1_gera3 = f1_score(y_train2, pred3, average = None)
df1_gera4 = f1_score(y_train2, pred4, average = None)
df1_gera5 = f1_score(y_train2, pred5, average = None)

In [32]:
print("LogisticRegression - Accuracy:", score,", F1_Score:", f1_geral," Precision :",ps1," Recall :",r1,  "\n F1-Classe",df1_geral)
print("SGDClassifier - Accuracy:", score2,", F1_Score:", f1_geral2," Precision :",ps2," Recall :",r2,  "\n F1-Classe",df1_gera2)
print("ComplementNB - Accuracy:", score3,", F1_Score:", f1_geral3," Precision :",ps3," Recall :",r3,  "\n F1-Classe",df1_gera3)
print("RandomForestClassifier - Accuracy:", score4,", F1_Score:",f1_geral4," Precision :",ps4," Recall :",r4,  "\n F1-Classe",df1_gera4)
print("SVM - Accuracy:", score5,", F1_Score:", f1_geral5," Precision :",ps5," Recall :",r5,  "\n F1-Classe",df1_gera5)

LogisticRegression - Accuracy: 0.8565737051792829 , F1_Score: 0.8184667309546769  Precision : 0.8883647798742138  Recall : 0.792276247848537 
 F1-Classe [0.90163934 0.73529412]
SGDClassifier - Accuracy: 0.8446215139442231 , F1_Score: 0.8250245777102512  Precision : 0.8240661534074707  Recall : 0.8260183591508893 
 F1-Classe [0.88358209 0.76646707]
ComplementNB - Accuracy: 0.7450199203187251 , F1_Score: 0.611905682257441  Precision : 0.8392339544513457  Recall : 0.6175057372346529 
 F1-Classe [0.83919598 0.38461538]
RandomForestClassifier - Accuracy: 0.7928286852589641 , F1_Score: 0.7085045560121493  Precision : 0.8678760077842647  Recall : 0.6897948938611589 
 F1-Classe [0.86528497 0.55172414]
SVM - Accuracy: 0.8725099601593626 , F1_Score: 0.8415153906866615  Precision : 0.8989419424850786  Recall : 0.8163726333907056 
 F1-Classe [0.91160221 0.77142857]


In [33]:
from imblearn.over_sampling import SMOTE 
sm = SMOTE(random_state=42)

In [34]:
X_res, y_res = sm.fit_resample(X_train, y_train2)

In [35]:
X_train = X_res
y_train2 = y_res

In [36]:
X_test, y_test2 = sm.fit_resample(X_test, y_test2)

In [37]:
from sklearn.model_selection import cross_val_predict
cvC = 10
classifier = LogisticRegression()
pred = cross_val_predict(classifier, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier2 = SGDClassifier()
pred2 = cross_val_predict(classifier2, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier3 = BernoulliNB()
pred3 = cross_val_predict(classifier3, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier4 = RandomForestClassifier()
pred4 = cross_val_predict(classifier4, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier5 = LinearSVC()
pred5 = cross_val_predict(classifier5, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.2s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   7 out of  10 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.1s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    1.3s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished


In [38]:
from sklearn.metrics import  f1_score, precision_score, recall_score
acc1 = accuracy_score(y_train2, pred)
acc2 = accuracy_score(y_train2, pred2)
acc3 = accuracy_score(y_train2, pred3)
acc4 = accuracy_score(y_train2, pred4)
acc5 = accuracy_score(y_train2, pred5)

In [39]:
ps1 = precision_score(y_train2, pred, average = 'macro')
r1  = recall_score(y_train2, pred, average = 'macro') 

ps2 = precision_score(y_train2, pred2, average = 'macro')
r2 = recall_score(y_train2, pred2, average = 'macro') 

ps3 = precision_score(y_train2, pred3, average = 'macro')
r3  = recall_score(y_train2, pred3, average = 'macro') 

ps4 = precision_score(y_train2, pred4, average = 'macro')
r4  = recall_score(y_train2, pred4, average = 'macro') 

ps5 = precision_score(y_train2, pred5, average = 'macro')
r5  = recall_score(y_train2, pred5, average = 'macro') 

In [40]:
f1_geral = f1_score(y_train2, pred, average = 'macro')
f1_gera2 = f1_score(y_train2, pred2, average = 'macro')
f1_gera3 = f1_score(y_train2, pred3, average = 'macro')
f1_gera4 = f1_score(y_train2, pred4, average = 'macro')
f1_gera5 = f1_score(y_train2, pred5, average = 'macro')

In [41]:
df1_geral = f1_score(y_train2, pred, average = None)
df1_gera2 = f1_score(y_train2, pred2, average = None)
df1_gera3 = f1_score(y_train2, pred3, average = None)
df1_gera4 = f1_score(y_train2, pred4, average = None)
df1_gera5 = f1_score(y_train2, pred5, average = None)

In [42]:
print("LogisticRegression - Accuracy:", score,", F1_Score:", f1_geral," Precision :",ps1," Recall :",r1,  "\n F1-Classe",df1_geral)
print("SGDClassifier - Accuracy:", score2,", F1_Score:", f1_geral2," Precision :",ps2," Recall :",r2,  "\n F1-Classe",df1_gera2)
print("ComplementNB - Accuracy:", score3,", F1_Score:", f1_geral3," Precision :",ps3," Recall :",r3,  "\n F1-Classe",df1_gera2)
print("RandomForestClassifier - Accuracy:", score4,", F1_Score:"," Precision :",ps4," Recall :",r4,  "\n F1-Classe",df1_gera2)
print("SVM - Accuracy:", score5,", F1_Score:", f1_geral5," Precision :",ps5," Recall :",r5,  "\n F1-Classe",df1_gera2)

LogisticRegression - Accuracy: 0.8565737051792829 , F1_Score: 0.8593403461267826  Precision : 0.8682742128337984  Recall : 0.8601190476190477 
 F1-Classe [0.8488746  0.86980609]
SGDClassifier - Accuracy: 0.8446215139442231 , F1_Score: 0.8250245777102512  Precision : 0.8888888888888888  Recall : 0.8869047619047619 
 F1-Classe [0.88271605 0.8908046 ]
ComplementNB - Accuracy: 0.7450199203187251 , F1_Score: 0.611905682257441  Precision : 0.8181818181818181  Recall : 0.7142857142857143 
 F1-Classe [0.88271605 0.8908046 ]
RandomForestClassifier - Accuracy: 0.7928286852589641 , F1_Score:  Precision : 0.8265433058622298  Recall : 0.7976190476190477 
 F1-Classe [0.88271605 0.8908046 ]
SVM - Accuracy: 0.8725099601593626 , F1_Score: 0.8415153906866615  Precision : 0.8504611330698286  Recall : 0.8392857142857143 
 F1-Classe [0.88271605 0.8908046 ]


In [43]:
print("LogisticRegression - Accuracy:", acc1,", F1_Score:", f1_geral, "\n",df1_geral)
print("SGDClassifier - Accuracy:", acc2,", F1_Score:", f1_gera2, "\n",df1_gera2)
print("ComplementNB - Accuracy:", acc3,", F1_Score:", f1_gera3, "\n",df1_gera3)
print("RandomForestClassifier - Accuracy:", acc4,", F1_Score:", f1_gera4, "\n",df1_gera4)
print("SVM - Accuracy:", acc5,", F1_Score:", f1_gera5, "\n",df1_gera5)

LogisticRegression - Accuracy: 0.8601190476190477 , F1_Score: 0.8593403461267826 
 [0.8488746  0.86980609]
SGDClassifier - Accuracy: 0.8869047619047619 , F1_Score: 0.8867603235419328 
 [0.88271605 0.8908046 ]
ComplementNB - Accuracy: 0.7142857142857143 , F1_Score: 0.6888888888888889 
 [0.6        0.77777778]
RandomForestClassifier - Accuracy: 0.7976190476190477 , F1_Score: 0.793035979564477 
 [0.76223776 0.8238342 ]
SVM - Accuracy: 0.8392857142857143 , F1_Score: 0.8379942140790742 
 [0.82352941 0.85245902]


# VALIDAÇÃO

In [44]:
import pandas as pd
strDir = "/Users/paulocordeiro/Dropbox/_MESTRADO/NOTEBOOKS/_WEBMIDIA 2019/GET_OLD_TWEETS/"
strFile = str(DATABASE_DIR / "FakeTweet" / "merge-val12-8.csv")
df_train = pd.read_csv(strFile)

FileNotFoundError: [Errno 2] No such file or directory: 'merge-val12-8.csv'

In [ ]:
df_train['classificacao'].value_counts()

In [ ]:
df_train['subject'].value_counts()

In [ ]:
sentences =  df_train['text'].values
y = df_train['classificacao'].values

In [ ]:
X_test, y_test2 = sm.fit_resample(X_test, y_test2)

In [ ]:
X_val = vectorizer.transform(sentences)

In [ ]:
y_val = transforma_vetor_str_int(y)

In [ ]:
classifier = LogisticRegression()
classifier.fit(X_train, y_train2)

classifier2 = SGDClassifier()
classifier2.fit(X_train, y_train2)

classifier3 = BernoulliNB()
classifier3.fit(X_train, y_train2)

classifier4 = RandomForestClassifier()
classifier4.fit(X_train, y_train2)

classifier5 = LinearSVC()
classifier5.fit(X_train, y_train2)

In [ ]:
score = classifier.score(X_val, y_val)
score2 = classifier2.score(X_val, y_val)
score3 = classifier3.score(X_val, y_val)
score4 = classifier4.score(X_val, y_val)
score5 = classifier5.score(X_val, y_val)

In [ ]:
from sklearn.metrics import  f1_score, precision_score, recall_score

y_predict_val = classifier.predict(X_val)
y_predict_val2 = classifier2.predict(X_val)
y_predict_val3 = classifier3.predict(X_val)
y_predict_val4 = classifier4.predict(X_val)
y_predict_val5 = classifier5.predict(X_val)



In [ ]:
ps1 = precision_score(y_val, y_predict_val, average = 'macro')
r1  = recall_score(y_val, y_predict_val, average = 'macro') 
ps2 = precision_score(y_val, y_predict_val2, average = 'macro')
r2 = recall_score(y_val, y_predict_val2, average = 'macro') 
ps3 = precision_score(y_val, y_predict_val3, average = 'macro')
r3  = recall_score(y_val, y_predict_val3, average = 'macro') 

ps4 = precision_score(y_val, y_predict_val4, average = 'macro')
r4  = recall_score(y_val, y_predict_val4, average = 'macro') 

ps5 = precision_score(y_val, y_predict_val5, average = 'macro')
r5  = recall_score(y_val, y_predict_val5, average = 'macro') 


In [ ]:
df1_geral = f1_score(y_val, y_predict_val, average = None)
df1_gera2 = f1_score(y_val, y_predict_val2, average = None)
df1_gera3 = f1_score(y_val, y_predict_val3, average = None)
df1_gera4 = f1_score(y_val, y_predict_val4, average = None)
df1_gera5 = f1_score(y_val, y_predict_val5, average = None)

f1_geral = f1_score(y_val, y_predict_val, average = 'macro')
f1_geral2 = f1_score(y_val, y_predict_val2, average = 'macro')
f1_geral3 = f1_score(y_val, y_predict_val3, average = 'macro')
f1_geral4 = f1_score(y_val, y_predict_val4, average = 'macro')
f1_geral5 = f1_score(y_val, y_predict_val5, average = 'macro')

In [ ]:
print("LogisticRegression - Accuracy:", score,", F1_Score:", f1_geral," Precision :",ps1," Recall :",r1,  "\n F1-Classe",df1_geral)
print("SGDClassifier - Accuracy:", score2,", F1_Score:", f1_geral2," Precision :",ps2," Recall :",r2,  "\n F1-Classe",df1_gera2)
print("ComplementNB - Accuracy:", score3,", F1_Score:", f1_geral3," Precision :",ps3," Recall :",r3,  "\n F1-Classe",df1_gera3)
print("RandomForestClassifier - Accuracy:", score4,", F1_Score:",f1_geral4," Precision :",ps4," Recall :",r4,  "\n F1-Classe",df1_gera4)
print("SVM - Accuracy:", score5,", F1_Score:", f1_geral5," Precision :",ps5," Recall :",r5,  "\n F1-Classe",df1_gera5)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_val, y_predict_val5)

In [ ]:
import pandas as pd
y_actu = pd.Series(y_val, name='Actual')
y_pred = pd.Series(y_predict_val5, name='Predicted')
df_confusion = pd.crosstab(y_actu, y_pred)

In [ ]:
df_conf_norm = df_confusion / df_confusion.sum(axis=1)

In [ ]:
df_conf_norm